## Models

- In the previous notebook, we transformed weak raw signals into structured features that allow non linear models to exploit regime dependent behavior discovered during EDA
- Now we focuse on training and evaluating several non-linear models using cross-validation.

In [44]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
DATA_PATH_RAW = "data/raw"
DATA_PATH_PROCESSED = "data/processed"
OUTPUT_FILE = "submission_final.csv"

In [45]:
X_train = pd.read_csv(f"{DATA_PATH_PROCESSED}/x_train_fe.csv")
y_train = pd.read_csv(f"{DATA_PATH_PROCESSED}/y_train_fe.csv")
X_test = pd.read_csv(f"{DATA_PATH_PROCESSED}/x_test_fe.csv")



In [46]:
def market_neutral_accuracy(df, scores):
    df = df.copy()
    df["score"] = scores

    # 1. Ta prédiction : Est-ce que je pense que ça va battre la médiane ?
    preds = df.groupby("DATE")["score"].transform(
        lambda x: x > x.median()
    ).astype(int)

    # 2. La réalité : Est-ce que ça a VRAIMENT battu la médiane ce jour-là ?
    # (C'est la vraie métrique du challenge)
    true_target = df.groupby("DATE")["RET"].transform(
        lambda x: x > x.median()
    ).astype(int)

    return accuracy_score(true_target, preds)

In [47]:
def cross_validate_model(model, X, y, df_meta, name):
    """
    Cross-validation for a binary classification model.
    Splits are done by DATE to preserve temporal structure.
    Predicted probabilities are converted to binary labels (0/1) for accuracy calculation.
    
    Parameters:
    - model: scikit-learn classifier with predict_proba
    - X: feature matrix (DataFrame)
    - y: binary target (Series)
    - df_meta: DataFrame containing meta information, must include "DATE"
    - name: model name (string) for printing
    
    Returns:
    - List of accuracy scores for each fold
    """

    dates = df_meta["DATE"].unique()  # all unique dates
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(dates)):
        # Select train and validation dates
        tr_dates, va_dates = dates[tr_idx], dates[va_idx]

        # Create boolean masks for train and validation indices
        tr_mask = df_meta["DATE"].isin(tr_dates)
        va_mask = df_meta["DATE"].isin(va_dates)

        # Fit the model on the training fold
        model.fit(X[tr_mask], y[tr_mask])

        # Predict probabilities on the validation fold
        scores_va = model.predict_proba(X[va_mask])[:, 1]

        # Convert probabilities to binary labels (0 or 1)
        scores_va_label = (scores_va > 0.5).astype(int)

        # Compute accuracy
        acc = accuracy_score(y[va_mask], scores_va_label)
        scores.append(acc)

        print(f"{name} | Fold {fold+1}: {acc:.4f}")

    mean_score = np.mean(scores)
    print(f"{name} | CV Mean Accuracy: {mean_score:.4f}\n")
    return scores



In [48]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lgbm = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


In [49]:
print("Cross-validation results:\n")

x_train_raw = pd.read_csv(f"{DATA_PATH_RAW}/x_train.csv")
y_train_raw = pd.read_csv(f"{DATA_PATH_RAW}/y_train.csv")

train = x_train_raw[['DATE']].copy()
train['RET'] = y_train_raw['RET']

score_rf = cross_validate_model(rf, X_train, y_train, train, "Random Forest")
score_xgb = cross_validate_model(xgb, X_train, y_train, train, "XGBoost")
score_lgbm = cross_validate_model(lgbm, X_train, y_train, train, "LightGBM")

scores = {
    "Random Forest": score_rf,
    "XGBoost": score_xgb,
    "LightGBM": score_lgbm
}

scores


Cross-validation results:



c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest | Fold 1: 0.5119


c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest | Fold 2: 0.5101


...

c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007946 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3573
[LightGBM] [Info] Number of data points in the train set: 334969, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.498933 -> initscore=-0.004269
[LightGBM] [Info] Start training from score -0.004269
LightGBM | Fold 4: 0.5164
[LightGBM] [Info] Number of positive: 167237, number of negative: 168107
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008242 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3573
[LightGBM] [Info] Number of data points in the train set: 335344, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.498703 -> initscore=-0.005189
[LightGBM] [Info] Start training from score -0.005189


c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


LightGBM | Fold 5: 0.5124
LightGBM | CV Mean Accuracy: 0.5132



{'Random Forest': [0.5118624452343556,
  0.5100893784083722,
  0.5171916672673539,
  0.51755434912587,
  0.5168706682202016],
 'XGBoost': [0.5094179194369324,
  0.5094569277216262,
  0.5184290828708042,
  0.5168727429268409,
  0.511008876770249],
 'LightGBM': [0.5100083846053921,
  0.5100655123447214,
  0.5172036810110767,
  0.5164183387941549,
  0.512354205955484]}

In [51]:
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
lgbm.fit(X_train, y_train)

ensemble_scores = (
    rf.predict_proba(X_train)[:, 1] +
    xgb.predict_proba(X_train)[:, 1] +
    lgbm.predict_proba(X_train)[:, 1]
) / 3

ensemble_labels = (ensemble_scores > 0.5).astype(int)

# Retrieve the True target
y_true = y_train.values.ravel()  

ensemble_acc = accuracy_score(y_true, ensemble_labels)

print(f"ENSEMBLE Accuracy: {ensemble_acc:.4f}")



c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\kraif\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Info] Number of positive: 208846, number of negative: 209749
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009273 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3573
[LightGBM] [Info] Number of data points in the train set: 418595, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.498921 -> initscore=-0.004314
[LightGBM] [Info] Start training from score -0.004314
ENSEMBLE Accuracy: 0.5831


In [ ]:
feature_cols = (
    [f"RET_{i}" for i in range(1, 6)] +
    [f"VOLUME_{i}" for i in range(1, 6)] +
    ["RET_mean_5", "RET_mean_10", "VOLATILITY_5", "VOLATILITY_10", "VOL_REGIME_5"]
)

X_test_fe = X_test[feature_cols].fillna(0)

X_test_fe = X_test_fe.copy()  
X_test_fe["ID"] = np.arange(len(X_test_fe))

# Predictions
test_scores = (
    rf.predict_proba(X_test_fe[feature_cols])[:, 1] +
    xgb.predict_proba(X_test_fe[feature_cols])[:, 1] +
    lgbm.predict_proba(X_test_fe[feature_cols])[:, 1]
) / 3

X_test_fe["score"] = test_scores

median_score = X_test_fe["score"].median()
X_test_fe["RET"] = X_test_fe["score"] > median_score

submission = X_test_fe[["ID", "RET"]]
submission.to_csv("submission_final.csv", index=False)

print("Submission saved to submission_final.csv")

Submission saved to submission_final.csv
